# AethyxLM v3 - prepare the 8B-token corpus on Kaggle

Run this notebook with a **CPU runtime and Internet enabled**. It streams raw datasets, applies the committed filters, tokenizes with the frozen 48K tokenizer, and writes exactly 8B `uint16` tokens (~14.9 GiB). The preparation is durable per source and resumes from its state files.

After an interruption or completion, run the upload cell. On a later session, attach that private Kaggle Dataset and rerun from the top.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook on Kaggle.')
REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'
REPO_ROOT = Path('/kaggle/working/aethyxlm-v3-repo')
PROJECT_ROOT = REPO_ROOT / 'AethyxLM'
DATA_ROOT = Path('/kaggle/working/aethyxlm-v3-data')
DATASET_HANDLE = 'aethyx/aethyxlm-v3-8b-tokenized'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
elif REPO_ROOT.exists():
    raise RuntimeError(f'{REPO_ROOT} exists but is not a Git checkout.')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tokenizers>=0.13', 'datasets>=2.14', 'kagglehub>=0.3',
                'tensorboard>=2.14', 'tqdm>=4.65', 'pyyaml>=6'], check=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUTF8'] = '1'
print('[OK] Project:', PROJECT_ROOT)
print('[OK] Data output:', DATA_ROOT)


In [ ]:
# Restore partial/completed preparation from any attached Kaggle Dataset.
manifest = json.loads((PROJECT_ROOT / 'configs/pretrain_8b_sources.json').read_text())
bundle = manifest['aethyxlm_v3_48k_8b']
expected_names = set()
for source in bundle['sources']:
    prefix = source['name']
    expected_names.update({
        f'{prefix}_train.bin', f'{prefix}_val.bin',
        f'{prefix}_train.bin.meta.json', f'{prefix}_val.bin.meta.json',
        f'{prefix}_metadata.json', f'{prefix}_state.json',
    })
input_root = Path('/kaggle/input')
restored = 0
for name in sorted(expected_names):
    candidates = [p for p in input_root.rglob(name) if p.is_file()]
    if not candidates:
        continue
    source = max(candidates, key=lambda p: p.stat().st_mtime)
    destination = DATA_ROOT / name
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
        restored += 1
print(f'[OK] Restored {restored} preparation artifacts from /kaggle/input.')


In [ ]:
# Validate the frozen tokenizer before spending hours on corpus preparation.
from tokenizer.tokenizer import AethyxTokenizer
tokenizer_path = PROJECT_ROOT / 'tokenizer/tokenizer_v3_48k.json'
tokenizer = AethyxTokenizer(tokenizer_path)
if tokenizer.vocab_size != 48000:
    raise RuntimeError(f'Expected 48,000 vocabulary entries, found {tokenizer.vocab_size}.')
print('[OK] Tokenizer v3:', tokenizer_path)
print('[OK] SHA-256:', tokenizer.sha256)


In [ ]:
# Prepare/resume all sources. Interrupting is safe after the current document flushes.
command = [
    sys.executable, 'scripts/prepare_dataset_bundle.py',
    '--manifest', 'configs/pretrain_8b_sources.json',
    '--bundle', 'aethyxlm_v3_48k_8b',
    '--output-dir', str(DATA_ROOT),
    '--buffer-tokens', '500000',
    '--progress-seconds', '30',
    '--registry-output', str(DATA_ROOT / 'datasets_v3_8b.json'),
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Persist the current state as a private Kaggle Dataset version.
import kagglehub
notes = 'AethyxLM v3 8B corpus preparation state'
kagglehub.dataset_upload(DATASET_HANDLE, str(DATA_ROOT), version_notes=notes)
print('[OK] Uploaded:', DATASET_HANDLE)


In [ ]:
# Completion audit: exactly 8B tokens and every source complete.
total_tokens = 0
incomplete = []
for source in bundle['sources']:
    prefix = source['name']
    metadata_path = DATA_ROOT / f'{prefix}_metadata.json'
    if not metadata_path.is_file():
        incomplete.append(prefix)
        continue
    metadata = json.loads(metadata_path.read_text())
    if metadata.get('status') != 'complete':
        incomplete.append(prefix)
        continue
    total_tokens += int(metadata['train_tokens']) + int(metadata['validation_tokens'])
print(f'Tokens: {total_tokens:,} / 8,000,000,000')
if incomplete:
    raise RuntimeError(f'Incomplete sources ({len(incomplete)}): {incomplete}')
if total_tokens != 8_000_000_000:
    raise RuntimeError('Prepared token total is not exactly 8B.')
print('[READY] Attach this private Dataset to kaggle_train_production.ipynb.')
